An asynchronous function (or coroutine) in Python is a function that can pause its execution while waiting for an external operation (like a network request or disk read) to finish, allowing the program to handle other tasks in the meantime.

#### Keywords

- async def: Declares a function as a coroutine. Calling an async def function does not immediately run its body; it returns a coroutine object.

- await: Pauses the execution of the coroutine until the awaited awaitable (another coroutine, Task, or Future) completes. It can only be used inside an async def function.

- Event Loop: The scheduler that coordinates execution. It runs tasks, manages I/O events, and switches between coroutines whenever an await yields control.

- asyncio.run(): The standard entry point that starts the event loop, runs the passed root coroutine until completion, and closes the loop.

- asyncio.gather(): Runs multiple coroutines concurrently and collects their results in order.

In [ ]:
import time

def fetch_data(task_id: int, delay: int):
    print(f"Task {task_id}: Started (waiting {delay}s)...")
    time.sleep(delay)  # ❌ BLOCKS the entire Python process
    print(f"Task {task_id}: Finished!")
    return f"Result {task_id}"

start_time = time.time()

# Sequential execution
fetch_data(1, 2)
fetch_data(2, 3)
fetch_data(3, 1)

print(f"Total time taken: {time.time() - start_time:.2f} seconds")

In [ ]:
import asyncio
import time

async def fetch_data_async(task_id: int, delay: int):
    print(f"Task {task_id}: Started (waiting {delay}s)...")
    # 'await' yields control back to the event loop while waiting
    await asyncio.sleep(delay)  
    print(f"Task {task_id}: Finished!")
    return f"Result {task_id}"

async def main():
    start_time = time.time()

    # Schedule and run all 3 coroutines concurrently on a single thread
    results = await asyncio.gather(
        fetch_data_async(1, 2),
        fetch_data_async(2, 3),
        fetch_data_async(3, 1)
    )

    print(f"Results: {results}")
    print(f"Total time taken: {time.time() - start_time:.2f} seconds")

# The entry point that spins up the Event Loop
await main()

## Async function use with LLM prompting for better performance

#### prompt running as a simple loop without using async function

In [ ]:
import time
from google import genai

client = genai.Client()

prompts = [
    "Explain gravity in one sentence.",
    "Give me a haiku about AI.",
    "Name three primary colors.",
    "What is the capital of Japan?",
]

def get_response(prompt: str) -> str:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    return response.text


start_time = time.perf_counter()
results = []

for idx, p in enumerate(prompts, start=1):
    print(f"[{idx}/{len(prompts)}] Sending: '{p}'")
    res = get_response(p)
    results.append(res)
    print(f"Response:\n{res.strip()}\n{'-'*40}")

elapsed = time.perf_counter() - start_time
print(f"Total Synchronous Time: {elapsed:.2f} seconds")




### Use Async for multiple LLM prompts in parallel

In [ ]:
import time
import asyncio
from google import genai

client = genai.Client()

prompts = [
    "Explain gravity in one sentence.",
    "Give me a haiku about AI.",
    "Name three primary colors.",
    "What is the capital of Japan?",
]

async def fetch_async(prompt_idx: int, prompt: str) -> str:
    print(f"[{prompt_idx}/{len(prompts)}] Sending: '{prompt}'")
    
    response = await client.aio.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    
    print(f"[{prompt_idx}/{len(prompts)}] Received response!")
    return response.text

# Measure execution directly in the notebook cell
start_time = time.perf_counter()

# Top-level await is natively supported in Jupyter
tasks = [fetch_async(idx, p) for idx, p in enumerate(prompts, start=1)]
results = await asyncio.gather(*tasks)

elapsed = time.perf_counter() - start_time

print("\n" + "=" * 40 + "\nAll Responses Received:\n" + "=" * 40)
for p, res in zip(prompts, results):
    print(f"Prompt: {p}")
    print(f"Response:\n{res.strip()}\n{'-'*40}")

print(f"Total Asynchronous Time: {elapsed:.2f} seconds")